# Session 1: why a car crumples
**A. Real-world problem.** A car hits a wall. The wall does not move, so the car must stop
in whatever distance the car itself has. If the front of the car is rigid, the car stops in
a few centimetres and the force is huge. If the front *crumples*, the car stops over half a
metre and the force is much smaller. Today you find the one relation behind that:

    F = m v^2 / (2 d)

**Plan.** (1) turn km/h into m/s, (2) compute the energy of a moving car, (3) turn that
energy into an average stopping force for a chosen crumple distance d, (4) plot force
against distance.

*This is a simplified educational model, not a crashworthiness design tool.*

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"
G = 9.81  # m/s^2, gravitational acceleration used to express forces as "g"

## Step 1: speed in the right unit
Physics wants metres per second. 1 km/h = 1000 m / 3600 s = 1/3.6 m/s.

In [ ]:
def kmh_to_ms(v_kmh):
    return v_kmh / 3.6

## Step 2: the energy that must go somewhere
A moving car carries kinetic energy KE = (1/2) m v^2 (joules). To stop, all of it must be
turned into something else: here, the work done by crushing metal.

In [ ]:
def kinetic_energy(m, v):
    """Kinetic energy in joules for mass m (kg) and speed v (m/s)."""
    return 0.5 * m * v * v

## Step 3: average force over a stopping distance
A force F acting over a distance d does work F*d. Setting work = kinetic energy:

    F * d = (1/2) m v^2     so     F = m v^2 / (2 d)

Doubling the distance halves the force. That is the whole idea of a crumple zone.

In [ ]:
def stopping_force(m, v, d):
    """Average force (newtons) needed to stop mass m (kg) moving at v (m/s) in distance d (m)."""
    if np.any(np.asarray(d) <= 0):
        raise ValueError("stopping distance must be positive")
    return m * v * v / (2.0 * d)


def deceleration_g(m, v, d):
    """The same stop, written as a deceleration in units of g (F = m a, so a = F / m)."""
    return stopping_force(m, v, d) / m / G

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt
    from matplotlib.patches import Rectangle

    OUT.mkdir(exist_ok=True)
    m, v = 1000.0, kmh_to_ms(50.0)
    print(f"speed {v:.2f} m/s, kinetic energy {kinetic_energy(m, v) / 1000:.1f} kJ")
    for d in (0.05, 0.2, 0.5, 1.0):
        print(f"stop in {d:4.2f} m: force {stopping_force(m, v, d) / 1000:7.1f} kN = {deceleration_g(m, v, d):6.1f} g")

    # Figure 1: force against stopping distance (a hyperbola)
    dist = np.linspace(0.05, 1.0, 200)
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(dist, stopping_force(m, v, dist) / 1000, color="C0")
    for d in (0.05, 0.2, 0.5):
        ax.plot(d, stopping_force(m, v, d) / 1000, "ko")
        ax.annotate(f"{stopping_force(m, v, d) / 1000:.0f} kN", (d, stopping_force(m, v, d) / 1000),
                    textcoords="offset points", xytext=(8, 4), fontsize=8)
    ax.set_xlabel("stopping distance d (m)")
    ax.set_ylabel("average stopping force F (kN)")
    ax.set_title("1000 kg car at 50 km/h: longer stop, smaller force")
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUT / "force_vs_distance.png", dpi=120)

    # Figure 2: the car-and-wall picture, rigid versus crumpling
    fig2, axs = plt.subplots(2, 1, figsize=(7, 4), sharex=True)
    for ax, (label, d) in zip(axs, [("rigid front: stops in 0.05 m", 0.05), ("crumple zone: stops in 0.50 m", 0.50)]):
        ax.add_patch(Rectangle((0, 0), 0.4, 1.0, color="0.7"))       # the wall
        ax.add_patch(Rectangle((0.4, 0.1), d, 0.8, color="C1", alpha=0.6))     # crushed length
        ax.add_patch(Rectangle((0.4 + d, 0.1), 3.5 - d, 0.8, color="C0", alpha=0.5))  # the rest of the car
        ax.set_xlim(0, 4.6)
        ax.set_ylim(0, 1)
        ax.set_yticks([])
        ax.set_title(f"{label}: force {stopping_force(m, v, d) / 1000:.0f} kN", fontsize=9)
    axs[1].set_xlabel("position along the road (m); orange = crushed length d")
    fig2.tight_layout()
    fig2.savefig(OUT / "car_wall.png", dpi=120)

## Coding tasks
1. **Easy.** Print the force and the g value for 1000 kg at 30, 50 and 70 km/h with a 0.5 m crumple distance. Predict first how the force changes between 30 and 70 km/h.
2. **Stretch.** Write `stopping_distance(m, v, F)` that returns the distance needed so the force does not exceed F, and use it to find the distance for 150 kN (it should match the answer to question D).
3. **Explain.** Explain in your own words why a rigid-front car stops 'harder' than a crumpling one even though both start with the same energy.

## Assessment
- **A. Conceptual.** A heavy truck and a small car hit a wall at the same speed and both stop in 0.5 m. Which has the larger average force? Which has the larger deceleration? Explain.
- **B. Numerical.** A 1500 kg car at 36 km/h is stopped in 0.4 m. Find the average force (in kN) and the deceleration (in g). Check your answer another way.
- **C. Graph and simulation interpretation.** On the force-versus-distance graph for a 1000 kg car at 50 km/h the force at d = 0.2 m is about 482 kN. Without a calculator, what is the force at d = 0.4 m and at d = 0.1 m? What is the curve called and why is it never zero?
- **D. Design and optimisation challenge.** Design: a 1000 kg car at 50 km/h must be stopped with an average force of at most 150 kN. What is the smallest crumple distance? Pick a practical value to the nearest 0.05 m and state the trade-off.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    checks.append(("36 km/h is 10 m/s", abs(L.kmh_to_ms(36.0) - 10.0) < 1e-12))
    checks.append(("kinetic energy of 1000 kg at 10 m/s is 50 kJ", abs(L.kinetic_energy(1000.0, 10.0) - 50000.0) < 1e-6))
    f = L.stopping_force(1000.0, 10.0, 0.5)
    checks.append(("hand calculation: 1000 kg, 10 m/s, 0.5 m needs 100 kN", abs(f - 100000.0) < 1e-6))
    checks.append(("doubling the distance halves the force", abs(L.stopping_force(1000.0, 10.0, 1.0) * 2 - f) < 1e-6))
    checks.append(("doubling the speed quadruples the force", abs(L.stopping_force(1000.0, 20.0, 0.5) / f - 4.0) < 1e-9))
    # worked example: 1000 kg at 50 km/h stopped in 0.5 m
    v = L.kmh_to_ms(50.0)
    fw = L.stopping_force(1000.0, v, 0.5)
    gw = L.deceleration_g(1000.0, v, 0.5)
    checks.append((f"worked example: 1000 kg at 50 km/h in 0.5 m gives {fw / 1000:.1f} kN = {gw:.1f} g",
                   abs(fw / 1000 - 192.9) < 0.1 and abs(gw - 19.7) < 0.05))
    checks.append(("deceleration in g does not depend on mass",
                   abs(L.deceleration_g(500.0, v, 0.5) - L.deceleration_g(2000.0, v, 0.5)) < 1e-9))
    try:
        L.stopping_force(1000.0, 10.0, 0.0)
        ok = False
    except ValueError:
        ok = True
    checks.append(("a zero stopping distance is rejected (it would need an infinite force)", ok))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
